# Batch Sensor Network Design for All HUC6 Basins in Texas-Gulf Region

This notebook:
1. Lists all HUC6 basins within Texas-Gulf (HUC2: 12)
2. For each HUC6 basin:
   - Downloads NWM streamflow data (skip if exists)
   - Designs optimal sensor network
   - Matches and evaluates USGS gauge network
   - Saves comparison results and visualizations
3. Generates summary statistics across all basins

In [1]:
from hydrosensenet.data import (
    NWMDataLoader, list_available_hucs,
    download_usgs_gauges, load_usgs_gauges, match_usgs_to_nwm
)
from hydrosensenet.io import save_streamflow, load_streamflow
from hydrosensenet import SensorNetworkDesigner
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import json
from datetime import datetime

# Configuration
HUC2 = "12"  # Texas-Gulf Region
YEARS = list(range(2000, 2010))  # 2000-2009 (10 years)
BASE_DIR = Path("./data/texas_gulf_huc6")
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Summary output
SUMMARY_FILE = BASE_DIR / "batch_summary.csv"
LOG_FILE = BASE_DIR / "batch_log.txt"

print(f"Batch Processing Configuration:")
print(f"  Region: HUC2 {HUC2} (Texas-Gulf)")
print(f"  Years: {YEARS[0]}-{YEARS[-1]} ({len(YEARS)} years)")
print(f"  Output: {BASE_DIR}")
print(f"  Summary: {SUMMARY_FILE}")

Batch Processing Configuration:
  Region: HUC2 12 (Texas-Gulf)
  Years: 2000-2009 (10 years)
  Output: data/texas_gulf_huc6
  Summary: data/texas_gulf_huc6/batch_summary.csv


## Step 1: Get All HUC6 Basins in Texas-Gulf

In [2]:
# List all HUC6 basins in Texas-Gulf region
print(f"Fetching HUC6 basins in region {HUC2}...\n")

huc6_list = list_available_hucs(region=HUC2, level=6)

print(f"✓ Found {len(huc6_list)} HUC6 basins in Texas-Gulf region\n")
print("HUC6 Basins:")
print(huc6_list.to_string(index=False))

# Store for batch processing
huc6_codes = huc6_list['huc'].tolist()
print(f"\nProcessing {len(huc6_codes)} basins: {huc6_codes}")

Fetching HUC6 basins in region 12...

✓ Found 22 HUC6 basins in Texas-Gulf region

HUC6 Basins:
   huc                       name states
120100                     Sabine  LA,TX
120200                     Neches     TX
120301              Upper Trinity     TX
120302              Lower Trinity     TX
120401                San Jacinto     TX
120402  Galveston Bay-Sabine Lake  LA,TX
120500          Brazos Headwaters  NM,TX
120601   Middle Brazos-Clear Fork     TX
120602       Middle Brazos-Bosque     TX
120701               Lower Brazos     TX
120702                     Little     TX
120800             Upper Colorado  NM,TX
120901     Middle Colorado-Concho     TX
120902      Middle Colorado-Llano     TX
120903             Lower Colorado     TX
120904        San Bernard Coastal     TX
121001                     Lavaca     TX
121002                  Guadalupe     TX
121003                San Antonio     TX
121004      Central Texas Coastal     TX
121101                     Nueces     TX
12

## Step 2: Download USGS Gauge Network (Once for All Basins)

In [ ]:
# Download USGS data once (shared across all basins)
usgs_geojson_file, usgs_comid_file = download_usgs_gauges(
    output_dir=BASE_DIR,
    huc2=None,  # All CONUS gauges
    force_download=False,
    verbose=True
)

usgs_gauges, comid_linkage = load_usgs_gauges(
    shapefile_path=usgs_geojson_file,
    comid_linkage_path=usgs_comid_file
)

print(f"\n✓ USGS data loaded: {len(usgs_gauges):,} stations")

## Step 3: Initialize NWM Data Loader

In [ ]:
# Initialize Dask-optimized loader (reusable for all basins)
loader = NWMDataLoader(
    version="3.0",
    use_dask=True,
    n_workers=6,
    threads_per_worker=4,
    memory_limit='2GB'
)
print("✓ NWM Data Loader initialized")

## Step 4: Batch Process All HUC6 Basins

In [ ]:
# Initialize summary results
batch_results = []

# Log file
def log_message(msg):
    """Write message to both console and log file"""
    print(msg)
    with open(LOG_FILE, 'a') as f:
        f.write(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - {msg}\n")

# Start batch processing
log_message(f"\n{'='*70}")
log_message(f"BATCH PROCESSING START: {len(huc6_codes)} basins")
log_message(f"{'='*70}\n")

for basin_idx, huc6 in enumerate(huc6_codes, 1):
    basin_name = huc6_list[huc6_list['huc'] == huc6]['name'].iloc[0]
    
    log_message(f"\n[{basin_idx}/{len(huc6_codes)}] Processing HUC6: {huc6} - {basin_name}")
    log_message(f"{'-'*70}")
    
    # Create basin-specific directory
    basin_dir = BASE_DIR / f"huc6_{huc6}"
    basin_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        # Check if this basin was already completed
        results_file = basin_dir / "results_summary.json"
        if results_file.exists():
            log_message(f"  ✓ Basin already processed - loading existing results")
            with open(results_file, 'r') as f:
                basin_result = json.load(f)
            batch_results.append(basin_result)
            continue
        
        # Filter by HUC6
        log_message(f"  [1/7] Filtering COMIDs for HUC6 {huc6}...")
        loader.filter_by_huc(huc6, huc_level=6)
        n_comids = len(loader.comids)
        log_message(f"        ✓ Found {n_comids:,} COMIDs")
        
        if n_comids == 0:
            log_message(f"        ⚠ No COMIDs found - skipping basin")
            continue
        
        # Download flowlines (skip if exists)
        log_message(f"  [2/7] Downloading flowlines...")
        flowlines_file = basin_dir / f"huc6_{huc6}_flowlines.geojson"
        if flowlines_file.exists():
            log_message(f"        ✓ Flowlines already exist")
            flowlines_gdf = gpd.read_file(flowlines_file)
        else:
            flowlines_file = loader.export_locations(
                output_file=flowlines_file,
                format="geojson",
                use_full_geometries=True
            )
            flowlines_gdf = gpd.read_file(flowlines_file)
            log_message(f"        ✓ Downloaded {len(flowlines_gdf):,} flowlines")
        
        # Download streamflow data (skip existing years)
        log_message(f"  [3/7] Downloading streamflow data...")
        missing_years = []
        for year in YEARS:
            year_file = basin_dir / f"streamflow_{year}.parquet"
            if not year_file.exists():
                missing_years.append(year)
        
        if missing_years:
            log_message(f"        Downloading {len(missing_years)} missing years: {missing_years}")
            loader.download_multiple_years(
                years=missing_years,
                output_dir=basin_dir,
                resample="D",
                resample_method="mean",
                batch_size=10000,
                format="parquet"
            )
            log_message(f"        ✓ Download complete")
        else:
            log_message(f"        ✓ All years already exist")
        
        # Load combined streamflow data
        log_message(f"  [4/7] Loading streamflow data...")
        dfs = []
        for year in YEARS:
            year_file = basin_dir / f"streamflow_{year}.parquet"
            dfs.append(load_streamflow(year_file))
        
        streamflow_combined = pd.concat(dfs, axis=0).sort_index()
        available_comids = [int(col) for col in streamflow_combined.columns]
        locations_gdf_filtered = flowlines_gdf[flowlines_gdf['comid'].isin(available_comids)].copy()
        
        log_message(f"        ✓ Loaded {streamflow_combined.shape[0]} timesteps × {streamflow_combined.shape[1]} locations")
        
        if streamflow_combined.shape[1] < 10:
            log_message(f"        ⚠ Too few locations ({streamflow_combined.shape[1]}) - skipping basin")
            continue
        
        # Match USGS gauges
        log_message(f"  [5/7] Matching USGS gauges...")
        location_labels = [str(c) for c in streamflow_combined.columns]
        usgs_comids, matched_usgs_gdf = match_usgs_to_nwm(
            usgs_gdf=usgs_gauges,
            comid_linkage=comid_linkage,
            available_comids=location_labels,
            verbose=False
        )
        
        if len(usgs_comids) > 0:
            usgs_indices = [location_labels.index(str(comid)) for comid in usgs_comids]
            n_sensors = len(usgs_indices)
            log_message(f"        ✓ Matched {n_sensors} USGS gauges")
        else:
            # Use default number if no USGS gauges
            n_sensors = min(50, streamflow_combined.shape[1] // 10)
            usgs_indices = []
            log_message(f"        ⚠ No USGS gauges matched - using {n_sensors} sensors")
        
        # Design optimal network
        log_message(f"  [6/7] Designing optimal sensor network ({n_sensors} sensors)...")
        designer = SensorNetworkDesigner(
            streamflow_data=streamflow_combined.values,
            locations=locations_gdf_filtered,
            location_labels=location_labels
        )
        
        optimal_result = designer.design_network(
            n_sensors=n_sensors,
            evaluate=True,
            verbose=False
        )
        
        log_message(f"        ✓ Optimal network: NNSE={np.nanmedian(optimal_result.performance_metrics['nnse']):.3f}")
        
        # Evaluate USGS network if available
        if usgs_indices:
            log_message(f"  [7/7] Evaluating USGS network...")
            usgs_result = designer.design_network(
                n_sensors=len(usgs_indices),
                existing_sensors=usgs_indices,
                evaluate=True,
                verbose=False
            )
            log_message(f"        ✓ USGS network: NNSE={np.nanmedian(usgs_result.performance_metrics['nnse']):.3f}")
            
            # Calculate improvement
            improvement = np.nanmedian(optimal_result.performance_metrics['nnse']) - \
                         np.nanmedian(usgs_result.performance_metrics['nnse'])
            
            # Save comparison plot
            plot_file = basin_dir / "comparison_plot.png"
            optimal_result.plot_comparison(
                baseline_result=usgs_result,
                flowlines_gdf=flowlines_gdf,
                location_labels=location_labels,
                comparison_labels={'optimal': 'Optimized', 'baseline': 'USGS'},
                metric='nnse',
                figsize=(7, 5),
                save_path=plot_file
            )
            plt.close('all')
        else:
            usgs_result = None
            improvement = None
            log_message(f"  [7/7] Skipping USGS evaluation (no gauges)")
        
        # Save results summary
        basin_result = {
            'huc6': huc6,
            'basin_name': basin_name,
            'n_comids': streamflow_combined.shape[1],
            'n_sensors': n_sensors,
            'n_usgs_gauges': len(usgs_indices),
            'optimal_nnse_median': float(np.nanmedian(optimal_result.performance_metrics['nnse'])),
            'optimal_nse_median': float(np.nanmedian(optimal_result.performance_metrics['nse'])),
            'usgs_nnse_median': float(np.nanmedian(usgs_result.performance_metrics['nnse'])) if usgs_result else None,
            'usgs_nse_median': float(np.nanmedian(usgs_result.performance_metrics['nse'])) if usgs_result else None,
            'improvement_nnse': float(improvement) if improvement is not None else None,
            'status': 'completed'
        }
        
        with open(results_file, 'w') as f:
            json.dump(basin_result, f, indent=2)
        
        batch_results.append(basin_result)
        log_message(f"  ✓ Basin {huc6} completed successfully")
        
    except Exception as e:
        log_message(f"  ✗ ERROR in basin {huc6}: {str(e)}")
        batch_results.append({
            'huc6': huc6,
            'basin_name': basin_name,
            'status': 'failed',
            'error': str(e)
        })
        continue

log_message(f"\n{'='*70}")
log_message(f"BATCH PROCESSING COMPLETE")
log_message(f"{'='*70}")

## Step 5: Generate Summary Report

In [ ]:
# Save summary to CSV
summary_df = pd.DataFrame(batch_results)
summary_df.to_csv(SUMMARY_FILE, index=False)

print(f"\n✓ Summary saved to: {SUMMARY_FILE}")
print(f"\nBatch Processing Summary:")
print(f"{'='*70}")
print(summary_df.to_string(index=False))

# Calculate statistics
completed = summary_df[summary_df['status'] == 'completed']
if len(completed) > 0:
    print(f"\n\nAggregate Statistics ({len(completed)} basins):")
    print(f"{'='*70}")
    print(f"Total COMIDs: {completed['n_comids'].sum():,}")
    print(f"Total sensors designed: {completed['n_sensors'].sum():,}")
    print(f"Total USGS gauges: {completed['n_usgs_gauges'].sum():,}")
    print(f"\nPerformance (median across basins):")
    print(f"  Optimal NNSE: {completed['optimal_nnse_median'].median():.3f}")
    print(f"  USGS NNSE: {completed['usgs_nnse_median'].median():.3f}")
    print(f"  Improvement (ΔNNSE): {completed['improvement_nnse'].median():.3f}")
    print(f"\nBasins with improvement > 0.1: {(completed['improvement_nnse'] > 0.1).sum()}")
else:
    print("\n⚠ No basins completed successfully")

## Step 6: Visualize Results Across Basins

In [ ]:
if len(completed) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: NNSE comparison
    ax = axes[0]
    x = np.arange(len(completed))
    width = 0.35
    
    ax.bar(x - width/2, completed['optimal_nnse_median'], width, label='Optimal', alpha=0.8)
    ax.bar(x + width/2, completed['usgs_nnse_median'], width, label='USGS', alpha=0.8)
    
    ax.set_xlabel('HUC6 Basin')
    ax.set_ylabel('Median NNSE')
    ax.set_title('Network Performance by Basin')
    ax.set_xticks(x)
    ax.set_xticklabels(completed['huc6'], rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    # Plot 2: Improvement distribution
    ax = axes[1]
    ax.hist(completed['improvement_nnse'].dropna(), bins=15, edgecolor='black', alpha=0.7)
    ax.axvline(0, color='red', linestyle='--', linewidth=2, label='No improvement')
    ax.set_xlabel('ΔNNSE (Optimal - USGS)')
    ax.set_ylabel('Number of Basins')
    ax.set_title('Improvement Distribution')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(BASE_DIR / 'batch_summary_plots.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\n✓ Summary plots saved to: {BASE_DIR / 'batch_summary_plots.png'}")
else:
    print("No data to visualize")